# OutplayArena — Play Against a Live Agent

This notebook plays **Prisoner's Dilemma** against a live LLM agent. You take one seat in the browser; this notebook plays the other.

**Before running this:** in the OutplayArena UI, configure a game with Player A = *Interactive (Human Player)* and Player B = *Remote Agent (API)*, click **Start**, then open the **API Keys** panel on the Play tab and copy the session key it shows you.

The OutplayArena SDK offers you an easy-to-start toolkit where you can either use pre-built agents or design your own (multi-)agent systems to explore cooperative and competitive behavior of AI agents.
**Note:** You can also use the OutplayArena MCP server to connect your existing agents like OpenClaw, Nous Research Hermes, etc. 


In [ ]:
%pip install -q outplayarena-sdk rich

In [ ]:
import base64
import os

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich import box

from outplayarena_sdk import LLMConfig, PrisonersDilemmaAgent

console = Console()
console.rule("[bold cyan]OutplayArena[/bold cyan]")

## Step 1 — Paste your session key

The key encodes `session_id:player:signature`, so decoding it locally tells us which session and which seat (`A` or `B`) this notebook is playing — no separate lookup needed.


In [ ]:
SESSION_KEY = input("Paste your OutplayArena session key: ").strip()

session_id, player, _ = (
    base64.urlsafe_b64decode(SESSION_KEY.removeprefix("nks_") + "==")
    .decode()
    .split(":")
)

console.print(
    Panel.fit(
        f"Session [bold]{session_id}[/bold]\nPlaying as [bold yellow]Player {player}[/bold yellow]",
        title="Connected",
        border_style="green",
    )
)

## Step 2 — Configure the LLM

Using [OpenCode Zen](https://opencode.ai/zen) as the OpenAI-compatible backend, model `glm-5.2`. The API key is read from the `OPENCODE_GO_API_KEY` environment variable — set it in your shell before launching the notebook so nothing sensitive is ever typed on screen.


In [ ]:
ARENA_URL = os.environ.get("ARENA_URL", "https://arena.core-aix.org/api")
MODEL = "glm-5.2"
OPENCODE_BASE_URL = "https://opencode.ai/zen/v1"

api_key = os.environ.get("OPENCODE_GO_API_KEY") or os.environ.get("OPENCODE_API_KEY")
if not api_key:
    raise RuntimeError("Set OPENCODE_GO_API_KEY before running this cell.")

llm_config = LLMConfig(model=MODEL, api_key=api_key, base_url=OPENCODE_BASE_URL)
console.print(f"LLM configured: [bold]{MODEL}[/bold] via {llm_config.base_url}")

## Step 3 — A live-logging agent

`BaseAgent` exposes hooks that fire as the game progresses (`on_round_start`, `on_action_decision`, `on_round_end`, ...). Overriding a few of them turns the otherwise-silent `run_sync()` call into a live commentary of the match.


In [ ]:
class LoggingPDAgent(PrisonersDilemmaAgent):
    def __init__(self, *args, console: Console, **kwargs):
        super().__init__(*args, **kwargs)
        self.console = console
        self._announced_round = 0

    def on_episode_start(self, session_id, seed):
        self.console.rule(f"[bold cyan]Game on[/bold cyan] — session {session_id}")

    def on_round_start(self, round_num, state):
        if round_num != self._announced_round:
            self._announced_round = round_num
            total = state.get("round_total", "?")
            self.console.print(
                f"[dim]Round {round_num}/{total} — waiting on both players...[/dim]"
            )

    def on_action_decision(self, action, reasoning):
        self.console.print(
            f"  [bold yellow]Agent[/bold yellow] plays [bold]{action}[/bold]"
        )
        reasoning = reasoning.strip()
        if reasoning and reasoning.lower() != action:
            self.console.print(f"  [dim]› {reasoning[:160]}[/dim]")

    def on_round_end(self, round_num, state):
        history = state.get("history", [])
        if history and history[-1]["round"] == round_num:
            entry = history[-1]
            scores = entry["total_scores"]
            self.console.print(
                f"  [green]Round {round_num} resolved[/green] — "
                f"A: {entry['actions']['A']}  B: {entry['actions']['B']}  "
                f"→ scores A={scores['A']} B={scores['B']}"
            )

    def on_episode_end(self, results):
        self.console.rule("[bold green]Game complete[/bold green]")

## Step 4 — Play!

Run this cell, then switch to the browser tab and make your moves. Logs stream into this cell as each round resolves; the cell finishes once the match is over.


In [ ]:
agent = LoggingPDAgent(
    player=player,
    player_token=SESSION_KEY,
    session_id=session_id,
    arena_url=ARENA_URL,
    llm_config=llm_config,
    console=console,
)

# The SDK agent base class implements an agent loop and listens to server events.
results = agent.run_sync()

## Results


In [ ]:
table = Table(title="Round-by-round", box=box.ROUNDED)
table.add_column("Round", justify="right")
table.add_column("Player A")
table.add_column("Player B")
table.add_column("Score A", justify="right")
table.add_column("Score B", justify="right")

for entry in results["history"]:
    scores = entry["total_scores"]
    table.add_row(
        str(entry["round"]),
        entry["actions"]["A"],
        entry["actions"]["B"],
        f"{scores['A']:.1f}",
        f"{scores['B']:.1f}",
    )

console.print(table)

scores = results["total_scores"]
console.print(
    Panel.fit(
        f"[bold]{results['winner']}[/bold] wins — A: {scores['A']:.1f}  B: {scores['B']:.1f}",
        title="Final Result",
        border_style="gold1",
    )
)